[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Exploring an API &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `doc` and
`resolve`. Run it first.


In [1]:
import json
import urllib.request
from pathlib import Path

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if not Path("practice_api.py").exists():      # true in Colab, which starts with only the notebook
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")

import practice_api

BASE = practice_api.start()

with urllib.request.urlopen(f"{BASE}/openapi.json") as response:
    doc = json.loads(response.read())


def resolve(doc, schema):
    """Return the schema a $ref points to, or the schema itself when there is no $ref."""
    if "$ref" not in schema:
        return schema
    node = doc
    for key in schema["$ref"].removeprefix("#/").split("/"):
        node = node[key]
    return node


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A response with its headers.


In [2]:
!curl -i {BASE}/stations/svalbard


HTTP/1.1 200 OK
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 77

{"id": "svalbard", "name": "Svalbard", "latitude": 78.22, "longitude": 15.65}

`Content-Length: 77` counts the bytes of the body. Svalbard's record is four bytes longer than
Tromso's, which is the extra length of its id and its name.


**2.** A method the endpoint does not allow.


In [3]:
!curl -i -X PATCH {BASE}/stations/oslo


HTTP/1.1 405 Method Not Allowed
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 58
Allow: GET

{"error": "PATCH not allowed: the stations are read-only"}

Only `GET` is allowed, because the stations are read-only. The status line and the `Allow` header
both say so, and the JSON body says why.


**3.** Every documented path and method.


In [4]:
for path, operations in doc["paths"].items():
    for method in operations:
        print(method.upper(), path)


GET /stations
GET /stations/{id}


The methods under a path are the keys of its dictionary, written in lowercase in an OpenAPI
document, which is why the loop calls `upper`.


**4.** The parameters of an operation.


In [5]:
for parameter in doc["paths"]["/stations/{id}"]["get"]["parameters"]:
    print(parameter["name"], parameter["in"], "required:", parameter["required"])


id path required: True


A path parameter is always required: without it, the path would not name a station at all.
OpenAPI makes this a rule, so every path parameter in any document says `required: true`.


**5.** The schema of an array's items.


In [6]:
schema = doc["paths"]["/stations"]["get"]["responses"]["200"]["content"]["application/json"]["schema"]
summary = resolve(doc, schema["items"])

print(schema["type"], "of", summary["type"])
print("required:", summary["required"])


array of object
required: ['id', 'name']


The array itself is written in place; only its items are a `$ref`. `resolve` returns a schema
unchanged when it holds no `$ref`, so it is safe to call on either.


**6.** Bergen, in Fahrenheit, with curl.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.


In [7]:
!curl -s -G https://archive-api.open-meteo.com/v1/archive \
    --data-urlencode latitude=60.39 --data-urlencode longitude=5.32 \
    --data-urlencode start_date=2025-01-15 --data-urlencode end_date=2025-01-17 \
    --data-urlencode daily=temperature_2m_mean --data-urlencode models=era5 \
    --data-urlencode temperature_unit=fahrenheit \
    -o bergen.json -w "status %{http_code}\n"


status 200


In [8]:
saved = Path("bergen.json")
bergen = json.loads(saved.read_text(encoding="utf-8"))
saved.unlink()

print(bergen["daily_units"]["temperature_2m_mean"], bergen["daily"]["temperature_2m_mean"])


°F [46.0, 46.3, 46.4]


Only the coordinates changed, and a seventh parameter was added. The `°F` in `daily_units` confirms
that Open-Meteo understood the unit, which is worth checking whenever a unit matters.


---

&#8592; **Back to:** [Exploring an API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/02-exploring-an-api.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
